In [1]:
import random
import numpy as np

from scipy.special import softmax






reward_1 = 100
reward_6 = 40

def step_left(state):
  if(state!=1):
    return state-1
  else:
    return -1

def step_right(state):
  if(state!=6):
    return state+1
  else:
    return -1

def step_null(state):
  return state

# print("\n\n")
# for i in range(1,7):
#   print(f" {step_left(i)} <- state {i} ")
#   print(f"state {i} -> {step_right(i)} ")
# print("\n\n")

def new_state(state,action):
  if(action == "L"):
    return step_left(state)
  elif(action == "R"):
    return step_right(state)
  else:
    return step_null(state)

def reward(state):
  if(state==1):
    return reward_1
  elif(state==6):
    return reward_6
  else:
    return 0


#Finds a trajectory from starting state, using return to calculate policy. 
#When returns is None,policy is calculated to be even distribution for actions, 1/3 for each since there are 3 actions
#when given a returns dictionary, it passes that dictionary to decide policy
#it doesnt calculate any return or even reward
#it can (and does), however, use updated policy to find optimal paths
#doesnt have its own returns, it takes return into account when given
def sampler(state,returns=None):
  trajectory = []
  s_init = state
  count = 0
  

  while(s_init!=1 and s_init!=6):

    if returns is None:
      action = random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init),k=1)[0]
    
    else:
      action = random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init,returns),k=1)[0]

    #print(f"current state: {s_init} action:{action} new state: {new_state(s_init,action)} count: {count} policy dist: {Policy.policy_dist_static(s_init)}")
    #print(f"{[s_init,action]}")
    trajectory.append([s_init,action,0,0])
    s_init = new_state(s_init,action)
    count = count + 1
    if((s_init!=6 and s_init!=1) and count>20000):
      return None

  trajectory.append([s_init,random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init),k=1)[0],0,0])
  
  return trajectory


#given a trajectory, calculate Gt for each state
def calculate_return(trajectory,discount=0.5):

  for i in range(len(trajectory) - 1, -1, -1):
    if(trajectory[i][0]==6 or trajectory[i][0]==1):
      trajectory[i][2] = reward(trajectory[i][0])
      trajectory[i][3] = trajectory[i][2]
    else:
      trajectory[i][3] = reward(trajectory[i][0]) + discount * trajectory[i+1][3]

  return trajectory

  #returned trajectory looks like [s,"action",reward,return]


#finds an average return value for each (s,a) combination
#This is the function that drives the learning.
# this function creates bunch of episodes and keeps track of the moving average of all the possible (s,a) combination.
#this returns a "returns" dictionary which later is used to decide policy function.

def return_estimation(start_state, num_episodes,rets=None,discount=0.5):
    returns = {}
    counts = {}

    for i in range(num_episodes):
      if rets is None:
        #("we are here")
        trajectory = sampler(start_state)
        #(trajectory)
      else:
        trajectory = sampler(start_state,rets)
        
      if trajectory is not None:
        #print(trajectory)
        trajectory = calculate_return(trajectory,discount=discount)
        for state, action, r, Gt in trajectory:
            key = (state, action)
            if key in returns:
                counts[key] += 1
                # update running average
                returns[key] += (Gt - returns[key]) / counts[key]
            else:
                returns[key] = Gt
                counts[key] = 1

    return returns




class Policy:
    zero_ret = {}
    def __init__(self,init_val=0):
        self.returns = {}
        self.policy_probability ={}
        a = ["L","0","R"]
        for i in range(1,7):
            for j in a:
                key = (i,j)
                self.returns[key] = init_val

        Policy.zero_ret = dict(self.returns) 
        for i in range(1,7):
            self.policy_probability[i] = self.policy_dist(i)

    def policy_dist(self,s,returns=None,):
        if(s==1 or s==6):
          return [0,1,0]
        dist =[]
        if returns == None:
          returns = self.returns
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)
        Qv = [Q1,Q2,Q3]
        dist = softmax(Qv) 
            
        return dist
    @staticmethod
    def policy_dist_static(s,returns=None):
        if(s==1 or s==6):
          return [0,1,0]
        dist =[]
        if returns == None:
          returns = Policy.zero_ret
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)
        Qv = [Q1,Q2,Q3]
        dist = softmax(Qv) 
            
        return dist
    
    def update(self,returns):
       self.returns = returns
       for i in range(1,7):
            self.policy_probability[i] = self.policy_dist(i)
    
    def update2(self,returns):
      returns_new = dict(self.returns)  # Start with a copy of old returns
      
      for (s,a), r_new in sorted(returns.items()):
        r_old = self.returns.get((s,a), 0)  # Default to 0 if not in old returns
        
        # Exponential moving average with alpha=0.9
        r_updated = r_old + (r_new - r_old) * 0.9
        returns_new[(s,a)] = r_updated

      self.returns = returns_new
      
      for i in range(1,7):
          self.policy_probability[i] = self.policy_dist(i)








In [2]:

# # We test our functions here.

# policy = Policy()

# # The following code segment basically shows the initial policy
# print("state:--- policy distribution:  L  0  R")
# for i in range(1,7):
#   print(f" {i}        policy distribution:  {Policy.policy_dist_static(i)[0]}  {Policy.policy_dist_static(i)[1]}  {Policy.policy_dist_static(i)[2]}")


# # Here we calculate a simple trajectory
# trajectory = sampler(4)

# # We find the return values for that trajectory
# trajectory = calculate_return(trajectory,discount=0.5)
# #We show the resulting values for this trajectory
# print("state   action   reward   return")
# for i in trajectory:
#   print(f"{i[0]}       {i [1]}        {i [2]}        {i [3]}")

# # We get different trajectories different times, different return values on different time too
# #Sometimes we dont even get all the states
# #this is basically a dice roll, a 3 sided one

In [3]:




# # count = 0
# # for i in range(2000):
# #    returns = return_estimation(4, 40)
# #    if(len(returns)!=14):
# #       count=count+1

# # print(count)

# #Our first encounter with returns. here we try to find the moving average of returns for each (s,a) combo by finding 40 trajectories 
# # and getting moving average of returns for (s,a)
# returns = return_estimation(4, 40,discount=0.995)

# print(len(returns))
# #len(returns) basically tells us how many (s,a) combo was found . there are 14 possible combo. since we dont assign any initial value for all (s,a) combo
# #We simply find enough trajectories so all the (s,a) shows up
# #40 almost always gives us all 14 trajectories, if its less than 40 then sometimes we dont discover all of them

# # Print results
# print("state   action   average_return")
# for (state, action), avg_return in sorted(returns.items()):
#     print(f"{state:<7}{action:<8}{avg_return:.6f}")


# #Here we see the relationship between policy and returns. After we got the returns, we pass it to the policy function to see a new distribution based on these returns

# print("state:--- policy distribution:  L  0  R")
# for i in range(1,7):
#   print(f" {i}        policy distribution:  {Policy.policy_dist_static(i,returns)[0]}  {Policy.policy_dist_static(i,returns)[1]}  {Policy.policy_dist_static(i,returns)[2]}")





Here is how our policy function works basically.

Our policy making has two parts as of now.

    1. expected returns
    2. proportional distribution

Initially we dont have a list of expected returns.
So our policy function takes this void of returns (not zeros,the VOID) and creates the equal distribution for actions.

We use this policy to calculate a list of expected returns. We find a trajectory, calculate it's returns and find another trajectory.
After running some iterations, we get a return list.

Next time we try to find a trajectory, we use this return find the distribution.

returns + policy_dist comes together to form the policy.

We cant calculate one without the other.

1. policy: return(null) -> distribution
2. distribution: -> trajectory
3. calculate_return: trajectory -> returns
4. policy: returns -> distribution
5. distribution: ->trajectory
6. repeat


Thoughts:

Right now we dont have a return tracking inherent to the policy functions. They are separated.

There is an initial return, None
This return is used to calculate trajectories by using policy_dist, which uses the return and in this case, None
The trajectories are all over the place but they gather info about expected return for each state+action
THIS trajectory is used to find a NEW return 
The NEW return is then used inside policy_dist when calculating another trajectory

We can do the following:

Create a class named policy
It will have inherent variable named expected_return
It will have a native policy distribution function that uses return to generate the distribution
When a state is passed into it for policy, it will give one
It can take entire trajectories to update its native expected_return values

In [4]:


# First we find expected return for (s,a) using the default or equalized policy distribution
# Then we PASS that expected return list to our policy function to redistribute the probability based on that return list
# This gives us a NEW return list, where actions that previously gained more return now gains even MORE return 
# This is again passed in the return calculator that again passes it to the policy function to redistribute based on this return
# This again makes actions with more return aggregate more
# This continues untill iteration ends



def naive_RL(iterations=100,discount=0.5,init_state=3):
  
  policyy = Policy()
  print(f"Starting naive RL with discount value:{discount} initial_state:{init_state}")

  print("state:--- policy distribution:  L  0  R BEFORE ITERATION")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policyy.policy_dist(i)[0]}  {policyy.policy_dist(i)[1]}  {policyy.policy_dist(i)[2]}")

  #convergence achieved at 100000
  returns = policyy.returns
  for i in range(iterations):
    
    returns = return_estimation(init_state, 50,returns,discount=discount)

    policyy.update2(returns)

  
  print(f"state:--- policy distribution:  L  0  R after all Iterations ")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policyy.policy_dist(i)[0]}  {policyy.policy_dist(i)[1]}  {policyy.policy_dist(i)[2]}")

  print("\n\n\n")
  
  return policyy





In [5]:




#THIS IS OUR OPTIMAL POLICY, or rather OUR DISTRIBUTION

policy = Policy()
policy = naive_RL(5000,discount=0.5,init_state=4)


print("After dust is settled")

print("Returns for discount 0.5")
for i in range(1,7):
    print(f" {i}        policy distribution:  {policy.policy_dist(i)[0]:0.6f}  {policy.policy_dist(i)[1]:0.6f}  {policy.policy_dist(i)[2]:0.6f}")


  
  

Starting naive RL with discount value:0.5 initial_state:4
state:--- policy distribution:  L  0  R BEFORE ITERATION
 1        policy distribution:  0  1  0
 2        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 3        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 4        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 5        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 6        policy distribution:  0  1  0
state:--- policy distribution:  L  0  R after all Iterations 
 1        policy distribution:  0  1  0
 2        policy distribution:  1.0  1.627070992350598e-19  4.768894683137928e-22
 3        policy distribution:  0.9999999998716729  8.244680594617267e-11  4.588049899900305e-11
 4        policy distribution:  0.9224984660646732  0.0017782486178593163  0.07572328531746746
 5        policy distribution:  9.29830293585582e-09  2.469226911

In [6]:
track = []
for i in range(100):
  tt = sampler(4,policy.returns)
  tt = [row[0] for row in tt]
  track.append(tt)


from collections import Counter
# Convert inner lists to tuples
track_tuples = [tuple(x) for x in track]
# Count occurrences
counts = Counter(track_tuples)
# Total number of elements
total = len(track)
# Calculate percentages
percentages = {key: (value / total) * 100 for key, value in counts.items()}
# Sort percentages by value descending and take top 3
top3 = sorted(percentages.items(), key=lambda x: x[1], reverse=True)[:]
# Print nicely
for element, pct in top3:
    print(f"{list(element)}: {pct:.2f}%")  # convert back to list if needed


[4, 3, 2, 1]: 90.00%
[4, 5, 6]: 9.00%
[4, 4, 3, 2, 1]: 1.00%


In [7]:
# def policy_random2(s,a):
#   if(s==6 or s==1):
#     return 1 if(a=="0") else 0
#   else:
#     return 1/3 if (a=="L") else 1/3 if(a=="R") else 1/3

# #instead of flat assignment, takes return values into account and assigns probability accordingly
# def modified_policy(s,a,rets):
#   Q1 = rets.get((s,"L"),0)
#   Q2 = rets.get((s,"0"),0)
#   Q3 = rets.get((s,"R"),0)
#   sum = Q1 + Q2 + Q3
#   if sum==0:
#     print("sum is zero")
#     return 1/3
#   pa_s = (Q1 if(a=="L") else Q2 if(a=="0") else Q3)/sum
#   return pa_s

# #random policy distribution for all actions for state s,  π(a_i|s) for a_i ∈ a, all actions
# def policy_dist_default(s):

#   return [policy_random2(s,"L"),policy_random2(s,"0"),policy_random2(s,"R")]

# #actual policy distribution for all actions for state s,  π(a_i|s) for a_i ∈ a, all actions, considering expected return
# def policy_dist(s,rets=None):
#   if rets is None:
#     return policy_dist_default(s)
#   return [modified_policy(s,"L",rets),modified_policy(s,"0",rets),modified_policy(s,"R",rets)]






# class Policy:
#     def __init__(self,init_val=0):
#         self.returns = {}
#         self.policy_probability ={}
#         a = ["L","0","R"]
#         for i in range(1,7):
#             for j in a:
#                 key = (i,j)
#                 self.returns[key] = init_val
        
#         for i in range(1,7):
#             self.policy_probability[i] = policy_dist(i)

#     def policy_dist(self,s):

#         dist =[]

#         Q1 = self.returns.get((s,"L"),0)
#         Q2 = self.returns.get((s,"0"),0)
#         Q3 = self.returns.get((s,"R"),0)
#         sum = Q1 + Q2 + Q3
#         if sum==0:
            
#             dist = [1/3,1/3,1/3]
#         else:
#            dist =[Q1/sum,Q2/sum,Q3/sum] 
            
#         return dist
#     def update(self,returns):
#        self.returns = returns
#        for i in range(1,7):
#             self.policy_probability[i] = self.policy_dist(i)



        
        


# pol = Policy()
# rets = pol.returns 
# props = pol.policy_probability


# for (state,action),avg_r in sorted(pol.returns.items()):
#     print(f"{state} {action} {avg_r}")

# # for i in range(1,7):
# #     print(f" {i}        policy distribution:  {Policy.policy_dist(i)[0]}  {Policy.policy_dist(i)[1]}  {Policy.policy_dist(i)[2]}")


# for s,p in sorted(props.items()):
#    print(f"state : {s} Distribution {p}")

# pol.update(returns)

# for (state,action),avg_r in sorted(pol.returns.items()):
#     print(f"{state} {action} {avg_r}")

# for s,p in sorted(props.items()):
#    print(f"state : {s} Distribution {p}")

In [8]:
import numpy as np

values = np.array([0,0,0])
probs = np.exp(values) / np.sum(np.exp(values))
print(probs)




[0.33333333 0.33333333 0.33333333]
